In [ ]:
!rm -rf /kaggle/working/Real-ESRGAN
!git clone --depth 1 --branch Kaggle https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
!pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
!cd /kaggle/working/Real-ESRGAN && python -m py_compile realesrgan.py realesrgan_fast.py realesrgan_fast_entry.py realesrgan_v5_entry.py enhance/*.py
!cd /kaggle/working/Real-ESRGAN && python realesrgan_v5_entry.py --help >/dev/null


In [ ]:
INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime/ts_2.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan.mp4"

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2
FPS = "source"

START_TIME = 3 * 60 + 15
TEST_SECONDS = 10
PROGRESS_INTERVAL = 60.0

# v5.0 quality policy: FP32 is enforced by the entry point.
FP16 = False
CHANNELS_LAST = True

AUTO_TILE = True
MAX_TILE_SIZE = 1536
AUTO_BATCH = True
MAX_BATCH_SIZE = 32
TILE_SIZE = 256
TILE_PAD = 10
TILE_VERIFY_COVERAGE = False
BATCH_SIZE = 4
GPU_IDS = "0,1"

# Conservative source-constrained dark-line restoration.
LINE_RESTORE = True
LINE_STRENGTH = 1.0
LINE_MAX_RECOVERY = 0.18
LINE_MAX_DARKENING = 0.10
LINE_MIN_CONTRAST = 0.025
LINE_EDGE_THRESHOLD = 0.010
LINE_ORIENTATION_FLOOR = 0.55

COLOR_POLICY = "preserve"
HDR_POLICY = "reject"

VIDEO_CODEC = "hevc_nvenc"
OUTPUT_PIX_FMT = "auto"
CRF = 18
PRESET = "medium"
CQ = 18
NVENC_PRESET = "p7"
SVTAV1_PRESET = 6
ENCODE_GPU = 0
AUTO_CODEC_FALLBACK = True

AUDIO_CODEC = "copy"
AUDIO_BITRATE = "192k"


In [ ]:
from collections import deque
import shlex
import subprocess
import sys

def boolean_flag(enabled, yes, no):
    return yes if enabled else no

def codec_probe(codec):
    probe = [
        "ffmpeg", "-hide_banner", "-loglevel", "error",
        "-f", "lavfi", "-i", "color=black:size=128x128:rate=1,format=rgb48le",
        "-frames:v", "1", "-c:v", codec,
    ]
    if codec in {"hevc_nvenc", "h264_nvenc"}:
        probe += [
            "-gpu", str(ENCODE_GPU), "-preset", NVENC_PRESET,
            "-tune", "hq", "-rc", "vbr", "-cq", str(CQ),
            "-b:v", "0", "-multipass", "fullres",
            "-spatial_aq", "1", "-temporal_aq", "1",
            "-rc-lookahead", "32", "-bf", "3",
            "-pix_fmt", "p010le" if codec == "hevc_nvenc" else "yuv420p",
        ]
    elif codec == "libsvtav1":
        probe += ["-preset", str(SVTAV1_PRESET), "-crf", str(CRF), "-pix_fmt", "yuv420p10le"]
    elif codec == "libx265":
        probe += ["-preset", PRESET, "-crf", str(CRF), "-pix_fmt", "yuv420p10le"]
    else:
        probe += ["-preset", PRESET, "-crf", str(CRF), "-pix_fmt", "yuv420p"]
    probe += ["-f", "null", "-"]
    result = subprocess.run(probe, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    return result.returncode == 0, result.stdout.strip()

effective_codec = VIDEO_CODEC
ok, codec_detail = codec_probe(effective_codec)
if not ok and AUTO_CODEC_FALLBACK:
    print(f"[encoder-warning] {effective_codec} probe failed; trying software fallback.", flush=True)
    if codec_detail:
        print(codec_detail, flush=True)
    for fallback in ("libx265", "libx264"):
        fallback_ok, _ = codec_probe(fallback)
        if fallback_ok:
            effective_codec = fallback
            print(f"[encoder] fallback selected: {effective_codec}", flush=True)
            break
    else:
        raise RuntimeError("No usable FFmpeg video encoder found.")
elif not ok:
    raise RuntimeError(f"Requested encoder {effective_codec} failed preflight:\n{codec_detail}")
else:
    print(f"[encoder] preflight OK: {effective_codec}", flush=True)

command = [
    sys.executable, "/kaggle/working/Real-ESRGAN/realesrgan_v5_entry.py",
    "--input", INPUT_VIDEO,
    "--output", OUTPUT_VIDEO,
    "--model", MODEL,
    "--model-path", MODEL_PATH,
    "--scale", str(SCALE),
    "--fps", str(FPS),
    boolean_flag(FP16, "--fp16", "--no-fp16"),
    boolean_flag(CHANNELS_LAST, "--channels-last", "--no-channels-last"),
    boolean_flag(AUTO_TILE, "--auto-tile", "--no-auto-tile"),
    "--max-tile-size", str(MAX_TILE_SIZE),
    boolean_flag(AUTO_BATCH, "--auto-batch", "--no-auto-batch"),
    "--max-batch-size", str(MAX_BATCH_SIZE),
    "--tile-size", str(TILE_SIZE),
    "--tile-pad", str(TILE_PAD),
    boolean_flag(TILE_VERIFY_COVERAGE, "--tile-verify-coverage", "--no-tile-verify-coverage"),
    "--batch-size", str(BATCH_SIZE),
    "--gpu-ids", GPU_IDS,
    boolean_flag(LINE_RESTORE, "--line-restore", "--no-line-restore"),
    "--line-strength", str(LINE_STRENGTH),
    "--line-max-recovery", str(LINE_MAX_RECOVERY),
    "--line-max-darkening", str(LINE_MAX_DARKENING),
    "--line-min-contrast", str(LINE_MIN_CONTRAST),
    "--line-edge-threshold", str(LINE_EDGE_THRESHOLD),
    "--line-orientation-floor", str(LINE_ORIENTATION_FLOOR),
    "--color-policy", COLOR_POLICY,
    "--hdr-policy", HDR_POLICY,
    "--video-codec", effective_codec,
    "--output-pix-fmt", OUTPUT_PIX_FMT,
    "--crf", str(CRF),
    "--preset", PRESET,
    "--cq", str(CQ),
    "--nvenc-preset", NVENC_PRESET,
    "--svtav1-preset", str(SVTAV1_PRESET),
    "--encode-gpu", str(ENCODE_GPU),
    "--audio-codec", AUDIO_CODEC,
    "--audio-bitrate", AUDIO_BITRATE,
    "--start-time", str(START_TIME),
    "--test-seconds", str(TEST_SECONDS),
    "--progress-interval", str(PROGRESS_INTERVAL),
    "--ffmpeg-bin", "ffmpeg",
    "--ffprobe-bin", "ffprobe",
]

print("[command]", shlex.join(command), flush=True)
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
tail = deque(maxlen=160)
for line in process.stdout:
    tail.append(line.rstrip())
    print(line, end="", flush=True)
return_code = process.wait()
if return_code != 0:
    child_tail = "\n".join(tail)
    raise RuntimeError(
        f"Real-ESRGAN v5.0 exited with code {return_code}.\n"
        f"--- child-process tail ---\n{child_tail}"
    )
